# 📊 Smart Retail Analytics - Exploratory Data Analysis (EDA)

**Dataset:** [Sample Superstore Retail Dataset](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)  
**Currency Calibration:** Converted from USD ($) to Indian Rupees (₹) at operational benchmark **1 USD = ₹95.28**  
**Objective:** Ingest, inspect, audit, clean, and explore historical retail transaction records to uncover revenue drivers, regional margin distributions, the *Discount Trap*, and customer purchasing patterns.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Visual styling configurations
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

# Currency calibration constant
USD_TO_INR = 95.28

## 1. Data Ingestion & Data Hygiene Audit
Load raw transaction dataset and audit schema, row count, null values, and data types.

In [ ]:
data_path = Path("../data/raw/SampleSuperstore.csv")
df_raw = pd.read_csv(data_path, encoding="latin-1")

print(f"Raw Records: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Missing Values Total: {df_raw.isnull().sum().sum()}")
print(f"Duplicate Orders: {df_raw.duplicated().sum()}")
df_raw.head(3)

## 2. Preprocessing & Currency Calibration (INR Conversion)
- Parse transaction and shipping timestamps
- Remove duplicate records
- Convert financial metrics from USD ($) to Indian Rupees (₹) at ₹95.28 / USD
- Derive helper attributes: `Year`, `Month`, `Quarter`, `Profit_Margin %`, and `Revenue_Per_Unit`

In [ ]:
df = df_raw.copy()
df["Order Date"] = pd.to_datetime(df["Order Date"])
df["Ship Date"] = pd.to_datetime(df["Ship Date"])
df = df.drop_duplicates()

# Currency conversion to INR
df["Sales"] = df["Sales"] * USD_TO_INR
df["Profit"] = df["Profit"] * USD_TO_INR

# Feature derivations
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Quarter"] = df["Order Date"].dt.to_period("Q").astype(str)
df["Profit_Margin"] = np.where(df["Sales"] > 0, (df["Profit"] / df["Sales"]) * 100, 0.0)
df["Revenue_Per_Unit"] = np.where(df["Quantity"] > 0, df["Sales"] / df["Quantity"], 0.0)

print(f"Cleaned Transactions: {len(df):,}")
print(f"Total Revenue (INR):  ₹{df['Sales'].sum():,.2f}")
print(f"Total Profit (INR):   ₹{df['Profit'].sum():,.2f}")
print(f"Average Profit Margin: {df['Profit_Margin'].mean():.2f}%")

## 3. Category & Regional Revenue vs Profitability
Evaluate revenue generation, net profitability, and average operating margins across the three product categories.

In [ ]:
cat_summary = df.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Avg_Margin=("Profit_Margin", "mean"),
    Orders=("Order ID", "nunique")
).reset_index()

cat_summary["Margin_Pct"] = (cat_summary["Total_Profit"] / cat_summary["Total_Sales"]) * 100
display(cat_summary)

## 4. The "Discount Trap" Analysis
Examine how discounting strategies impact gross margin. While discounts >20% increase short-term sales volume, they frequently inflict catastrophic negative margins in high-cost sub-categories.

In [ ]:
df["Discount_Tier"] = pd.cut(
    df["Discount"],
    bins=[-0.01, 0.0, 0.1, 0.2, 0.4, 1.0],
    labels=["0% (No Discount)", "1-10%", "11-20%", "21-40%", ">40%"]
)

discount_analysis = df.groupby("Discount_Tier", observed=False).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Avg_Margin=("Profit_Margin", "mean"),
    Transactions=("Row ID", "count")
).reset_index()

display(discount_analysis)

## 5. Monthly Sales Velocity & Seasonality
Aggregate time series monthly revenue to analyze momentum, trajectory, and fourth-quarter holiday sales spikes.

In [ ]:
df["YearMonth"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()
monthly = df.groupby("YearMonth").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum")
).reset_index()

print("Monthly Aggregation Sample (Last 6 Months):")
display(monthly.tail(6))

## 6. Customer Behavioral RFM Profile
Derive Recency (days since last purchase), Frequency (distinct order count), and Monetary spend (lifetime INR revenue) for each customer.

In [ ]:
snapshot_date = df["Order Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer ID").agg(
    Recency=("Order Date", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Order ID", "nunique"),
    Monetary=("Sales", "sum")
).reset_index()

print(f"Unique Customer Entities: {len(rfm)}")
display(rfm.describe().T.round(2))

## 7. Strategic Conclusions & Commercial Recommendations
1. **Enforce Strict 20% Discount Ceiling:** Excessive markdowns (>20%) destroy profitability, particularly on bulky furniture (Tables, Bookcases).
2. **Implement Q4 Inventory Buffers:** Predictable late-year demand surges require +15% safety stock starting in September.
3. **VIP Customer Retention Over Mass Promotions:** The top 20% of customers generate disproportionate lifetime value, warranting specialized loyalty privileges.